In [1]:

# # ── Inspect test-split videos: duration / fps / resolution ────────────
# import subprocess, json, re
# from pathlib import Path

# test_dir = Path.cwd() / "MELD.Raw" / "test" / "output_repeated_splits_test"
# videos = sorted(test_dir.glob("*.mp4"))
# print(f"Test videos found: {len(videos)}")

# STANDARD_RE = re.compile(r'^dia\d+_utt\d+\.mp4$')

# rows = []
# for p in videos:
#     is_standard = bool(STANDARD_RE.match(p.name))
#     try:
#         r = subprocess.run(
#             ["ffprobe", "-v", "quiet", "-print_format", "json",
#              "-show_streams", "-select_streams", "v:0", str(p)],
#             capture_output=True, text=True, timeout=5,
#         )
#         streams = json.loads(r.stdout).get("streams", [])
#         if streams:
#             s = streams[0]
#             dur  = float(s.get("duration", 0) or 0)
#             w, h = s.get("width"), s.get("height")
#             rn, rd = s.get("r_frame_rate", "0/1").split("/")
#             fps  = round(int(rn) / max(int(rd), 1), 3)
#             nf   = s.get("nb_frames", "?")
#             rows.append({"name": p.name, "standard": is_standard, "dur": dur, "fps": fps,
#                          "w": w, "h": h, "nb_frames": nf, "size_mb": round(p.stat().st_size / 1e6, 3)})
#         else:
#             rows.append({"name": p.name, "standard": is_standard, "dur": 0.0, "fps": None,
#                          "w": None, "h": None, "nb_frames": "no-stream", "size_mb": round(p.stat().st_size / 1e6, 3)})
#     except Exception as e:
#         rows.append({"name": p.name, "standard": is_standard, "dur": -1, "fps": None,
#                      "w": None, "h": None, "nb_frames": str(e), "size_mb": 0})

# import pandas as pd
# df_inspect = pd.DataFrame(rows).sort_values("dur", ascending=False)

# # ── Naming summary ─────────────────────────────────────────────────────
# std_count   = df_inspect["standard"].sum()
# nonst_count = (~df_inspect["standard"]).sum()
# print(f"\nNaming:  standard={std_count}  non-standard={nonst_count}")

# # Show non-standard names (first 10)
# nonstandard = df_inspect[~df_inspect["standard"]]
# print(f"\nNon-standard filenames ({len(nonstandard)} total) — first 10:")
# print(nonstandard["name"].head(10).to_string(index=False))

# # ── Duration summary ───────────────────────────────────────────────────
# durs = df_inspect[df_inspect["dur"] > 0]["dur"]
# print(f"\nDuration stats ({len(durs)} videos with valid duration):")
# print(f"  Max:    {durs.max():.2f}s")
# print(f"  Min:    {durs.min():.2f}s")
# print(f"  Mean:   {durs.mean():.2f}s")
# print(f"  Median: {durs.median():.2f}s")

# bins   = [0, 5, 10, 20, 30, 60, float("inf")]
# labels = ["≤5s", "5-10s", "10-20s", "20-30s", "30-60s", ">60s"]
# print("\nDistribution:")
# for lo, hi, lbl in zip(bins, bins[1:], labels):
#     n = ((durs > lo) & (durs <= hi)).sum()
#     print(f"  {lbl:>8}: {n}")

# no_dur = df_inspect[df_inspect["dur"] <= 0]
# print(f"\nVideos with no/zero duration: {len(no_dur)}")

# print("\nTop 20 longest:")
# print(df_inspect[["name", "standard", "dur", "fps", "w", "h", "nb_frames", "size_mb"]].head(20).to_string(index=False))


In [2]:

import os
import cv2
import torch
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

BASE_DIR = Path.cwd()
MELD_DIR = BASE_DIR / "MELD.Raw"
MANIFEST_PATH = MELD_DIR / "meld_vlm_manifest.csv"

print("Imports OK")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


2026-03-03 03:30:33.371177: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-03 03:30:33.404582: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Imports OK
CUDA available: True
GPU: NVIDIA GeForce RTX 3060
VRAM: 12.5 GB


In [3]:

MODEL_ID     = "KlingTeam/VidEmo-3B"
PROCESSOR_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

print(f"Loading model: {MODEL_ID}")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    # attn_implementation="flash_attention_2",  # enable if flash-attn is installed
)
model.eval()

processor = AutoProcessor.from_pretrained(PROCESSOR_ID)
# Decoder-only models require left-padding for correct batched generation
processor.tokenizer.padding_side = "left"

print("Model loaded successfully")


`torch_dtype` is deprecated! Use `dtype` instead!


Loading model: KlingTeam/VidEmo-3B


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


Model loaded successfully


In [4]:

# ── Build manifest from train / dev / test CSVs ───────────────────────
# Video path conventions (test dir has two naming styles):
#   standard : dia{D}_utt{U}.mp4
#   prefixed : final_videos_testdia{D}_utt{U}.mp4

SPLIT_CFG = {
    "train": {
        "csv":     MELD_DIR / "train" / "train_sent_emo.csv",
        "vid_dir": MELD_DIR / "train" / "train_splits",
    },
    "dev": {
        "csv":     MELD_DIR / "dev_sent_emo.csv",
        "vid_dir": MELD_DIR / "dev" / "dev_splits_complete",
    },
    "test": {
        "csv":     MELD_DIR / "test_sent_emo.csv",
        "vid_dir": MELD_DIR / "test" / "output_repeated_splits_test",
    },
}


def resolve_video_path(vid_dir: Path, dia: int, utt: int) -> str:
    """Return path to clip, trying standard name then the final_videos_test prefix."""
    for stem in (
        f"dia{dia}_utt{utt}.mp4",
        f"final_videos_testdia{dia}_utt{utt}.mp4",
    ):
        p = vid_dir / stem
        if p.exists():
            return str(p)
    return ""   # genuinely missing


frames = []
for split, cfg in SPLIT_CFG.items():
    df = pd.read_csv(cfg["csv"])
    df = df.rename(columns={"Sr No.": "sr_no"})
    df["split"] = split
    df["path"]  = df.apply(
        lambda r: resolve_video_path(cfg["vid_dir"], int(r["Dialogue_ID"]), int(r["Utterance_ID"])),
        axis=1,
    )
    frames.append(df)

manifest = pd.concat(frames, ignore_index=True)
manifest["vlm_analysis"] = pd.NA

# Drop rows whose video file wasn't resolved
missing = manifest["path"] == ""
print(f"Missing video files: {missing.sum()} (will be skipped)")
manifest = manifest[~missing].reset_index(drop=True)

print(f"\nManifest built:")
print(f"  Train : {(manifest['split']=='train').sum()}")
print(f"  Dev   : {(manifest['split']=='dev').sum()}")
print(f"  Test  : {(manifest['split']=='test').sum()}")
print(f"  Total : {len(manifest)}")
manifest.head(3)


Missing video files: 1 (will be skipped)

Manifest built:
  Train : 9989
  Dev   : 1108
  Test  : 2610
  Total : 13707


,sr_no,Utterance,Speaker,Emotion,Sentiment,Dialogue_ID,Utterance_ID,Season,Episode,StartTime,EndTime,split,path,vlm_analysis
0,1,also I was the point person on my companys tr...,Chandler,neutral,neutral,0,0,8,21,"00:16:16,059","00:16:21,731",train,/mnt/Work/ML/Code/EmoRecVid/MELD.Raw/train/tra...,<NA>
1,2,You mustve had your hands full.,The Interviewer,neutral,neutral,0,1,8,21,"00:16:21,940","00:16:23,442",train,/mnt/Work/ML/Code/EmoRecVid/MELD.Raw/train/tra...,<NA>
2,3,That I did. That I did.,Chandler,neutral,neutral,0,2,8,21,"00:16:23,442","00:16:26,389",train,/mnt/Work/ML/Code/EmoRecVid/MELD.Raw/train/tra...,<NA>


In [5]:

# ── Save initial manifest (or reload if resuming) ─────────────────────
if MANIFEST_PATH.exists():
    saved = pd.read_csv(MANIFEST_PATH)
    # Merge already-computed analyses back in
    manifest = manifest.merge(
        saved[["path", "vlm_analysis"]].rename(columns={"vlm_analysis": "_saved"}),
        on="path", how="left",
    )
    manifest["vlm_analysis"] = manifest["_saved"].combine_first(manifest["vlm_analysis"])
    manifest = manifest.drop(columns=["_saved"])
    done_before = manifest["vlm_analysis"].notna().sum()
    print(f"Resumed from existing manifest — {done_before}/{len(manifest)} already done.")
else:
    manifest.to_csv(MANIFEST_PATH, index=False)
    print(f"New manifest saved → {MANIFEST_PATH}")

manifest.head(3)


Resumed from existing manifest — 13342/13707 already done.


,sr_no,Utterance,Speaker,Emotion,Sentiment,Dialogue_ID,Utterance_ID,Season,Episode,StartTime,EndTime,split,path,vlm_analysis
0,1,also I was the point person on my companys tr...,Chandler,neutral,neutral,0,0,8,21,"00:16:16,059","00:16:21,731",train,/mnt/Work/ML/Code/EmoRecVid/MELD.Raw/train/tra...,"The speaker begins with a neutral expression, ..."
1,2,You mustve had your hands full.,The Interviewer,neutral,neutral,0,1,8,21,"00:16:21,940","00:16:23,442",train,/mnt/Work/ML/Code/EmoRecVid/MELD.Raw/train/tra...,"The speaker begins with a neutral expression, ..."
2,3,That I did. That I did.,Chandler,neutral,neutral,0,2,8,21,"00:16:23,442","00:16:26,389",train,/mnt/Work/ML/Code/EmoRecVid/MELD.Raw/train/tra...,"The speaker begins with a neutral expression, ..."


In [6]:
# manifest["vlm_analysis"] = pd.NA  # Clear any existing analyses before processing
# manifest.to_csv(MANIFEST_PATH, index=False)

In [ ]:

total       = len(manifest)
done_before = manifest["vlm_analysis"].notna().sum()
print(f"Total clips : {total}")
print(f"Already done: {done_before}  |  Remaining: {total - done_before}")

# ── Config ────────────────────────────────────────────────────────────
BATCH_SIZE      = 2    # clips per batch
SAVE_EVERY      = 2   # save checkpoint every N clips
MAX_FRAMES      = 16   # cap for long clips (>15 s)
LONG_CLIP_SECS  = 15   # threshold in seconds
FPS_SHORT       = 1.0  # target for short clips (1 frame/s expressed as nframes)

PROMPT_TEXT = (
    "Watch this short video clip and focus exclusively on the person who is actively speaking.\n"
    "Ignore any other people visible in the frame — describe only the speaker's emotional state "
    "as it evolves over the duration of the clip.\n"
    "Structure your response as a temporal progression — divide the clip into beginning, middle, and end "
    "(or more segments if the speaker's emotion changes more than once).\n"
    "For each segment describe:\n"
    "- **Facial Expressions**: Specific muscle movements of the speaker (brow, lips, eyes, jaw, cheeks).\n"
    "- **Head & Gaze**: The speaker's head tilts, nods, shakes, eye direction.\n"
    "- **Body Language**: The speaker's posture shifts, gestures, tension or relaxation.\n"
    "- **Emotion at this moment**: The most likely emotion of the speaker and the visual cues supporting it.\n"
    "Finish with a one-sentence summary of the speaker's overall emotional arc "
    "(e.g., starts neutral → builds frustration → brief smile at end).\n"
    "Ground every observation in a specific visible signal from the speaking person only."
)


def _clip_nframes(video_path: str) -> int:
    """Return how many frames to sample.
    - clip > LONG_CLIP_SECS  → MAX_FRAMES (uniformly spread)
    - clip ≤ LONG_CLIP_SECS  → ~1 frame/s, min 2 (torchvision requires nframes >= 2)
    Falls back to MAX_FRAMES when cv2 reports zero total frames (unreliable metadata).
    """
    cap = cv2.VideoCapture(video_path)
    fps_cv  = cap.get(cv2.CAP_PROP_FPS) or 25.0
    n_total = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    cap.release()

    # cv2 can return 0 for some containers — fall back to a safe default
    if n_total <= 0:
        return MAX_FRAMES

    duration = n_total / fps_cv if fps_cv > 0 else 0.0
    if duration > LONG_CLIP_SECS:
        return MAX_FRAMES
    # torchvision requires nframes >= 2
    return max(2, round(duration * FPS_SHORT))


def analyze_batch(video_paths: list[str], max_new_tokens: int = 512) -> list[str]:
    """Run VidEmo on a batch of video clips; returns one response string per clip."""
    all_messages = [
        [{"role": "user", "content": [
            {"type": "video", "video": vp,
             "nframes": _clip_nframes(vp),   # dynamic: ≤15s → ~1fps (min 2), >15s → 16 frames
             "max_pixels": 360 * 420},
            {"type": "text", "text": PROMPT_TEXT},
        ]}]
        for vp in video_paths
    ]

    texts = [
        processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        for msgs in all_messages
    ]
    image_inputs, video_inputs, video_kwargs = process_vision_info(
        all_messages, return_video_kwargs=True
    )
    if "fps" in video_kwargs and isinstance(video_kwargs["fps"], list):
        del video_kwargs["fps"]

    inputs = processor(
        text=texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
        **video_kwargs,
    ).to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    return [
        processor.decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()
        for ids in generated_ids_trimmed
    ]


# ── Main loop ──────────────────────────────────────────────────────────
pending = list(manifest[manifest["vlm_analysis"].isna()].index)
print(f"Queued: {len(pending)} clips\n")

processed = 0
with tqdm(total=len(pending), desc="VLM analysis") as pbar:
    for batch_start in range(0, len(pending), BATCH_SIZE):
        batch_idx   = pending[batch_start : batch_start + BATCH_SIZE]
        video_paths = [manifest.at[i, "path"] for i in batch_idx]

        try:
            results = analyze_batch(video_paths)
            for idx, result in zip(batch_idx, results):
                manifest.at[idx, "vlm_analysis"] = result
        except Exception as e:
            for idx in batch_idx:
                manifest.at[idx, "vlm_analysis"] = f"ERROR: {e}"
            print(f"\nError on batch {[Path(p).name for p in video_paths]}: {e}")

        torch.cuda.empty_cache()
        processed += len(batch_idx)
        pbar.update(len(batch_idx))

        if processed % SAVE_EVERY < BATCH_SIZE:
            manifest.to_csv(MANIFEST_PATH, index=False)

# Final save
manifest.to_csv(MANIFEST_PATH, index=False)

done   = manifest["vlm_analysis"].notna().sum()
errors = manifest["vlm_analysis"].str.startswith("ERROR:", na=False).sum()
print(f"\nDone. {done}/{total}  |  {errors} errors.")
print(f"Results saved → {MANIFEST_PATH}")
manifest.head(3)


Total clips : 13707
Already done: 13342  |  Remaining: 365
Queued: 365 clips



VLM analysis:   0%|          | 0/365 [00:00<?, ?it/s]qwen-vl-utils using torchvision to read video.
/mnt/Work/Environments/Ubuntu/Conda/envs/kaggle/lib/python3.12/site-packages/torchvision/io/_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(
VLM analysis:   1%|          | 4/365 [00:05<07:46,  1.29s/it]


Error on batch ['dia241_utt3.mp4', 'dia241_utt4.mp4', 'dia241_utt5.mp4', 'dia241_utt6.mp4']: CUDA out of memory. Tried to allocate 96.00 MiB. GPU 0 has a total capacity of 11.63 GiB of which 119.19 MiB is free. Process 304668 has 7.16 GiB memory in use. Including non-PyTorch memory, this process has 4.00 GiB memory in use. Of the allocated memory 3.82 GiB is allocated by PyTorch, and 55.25 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


VLM analysis:   1%|          | 4/365 [01:03<1:35:51, 15.93s/it]


KeyboardInterrupt: 

In [ ]:

# ── Retry errored / bad / empty clips ────────────────────────────────
import cv2
import torch
import pandas as pd
from pathlib import Path
from tqdm import tqdm

manifest = pd.read_csv(MANIFEST_PATH)


def _is_bad_output(val) -> bool:
    if pd.isna(val):
        return False
    s = str(val)
    if s.startswith("ERROR:") or s.startswith("SKIP:"):
        return False
    words = s.strip().split()
    if len(words) <= 10:
        return True
    if words[0].lower() == "the" and words[-1].lower() == "the":
        return True
    return False


pass_num = 0

while True:
    is_skip  = manifest["vlm_analysis"].str.startswith("SKIP:", na=False)
    is_error = manifest["vlm_analysis"].str.startswith("ERROR:", na=False)
    is_empty = manifest["vlm_analysis"].isna()
    is_bad   = manifest["vlm_analysis"].map(_is_bad_output)
    retry_idx = list(manifest[(is_error | is_empty | is_bad) & ~is_skip].index)

    print(f"\n── Pass {pass_num} ── "
          f"Empty: {is_empty.sum()}  |  Errored: {is_error.sum()}  |  "
          f"Bad: {is_bad.sum()}  |  Skipped: {is_skip.sum()}  |  "
          f"To retry: {len(retry_idx)}")

    if not retry_idx:
        print("All clips are clean — done!")
        break

    pass_num += 1
    processed = 0

    with tqdm(total=len(retry_idx), desc=f"Retry pass {pass_num}") as pbar:
        for batch_start in range(0, len(retry_idx), BATCH_SIZE):
            batch_idx   = retry_idx[batch_start : batch_start + BATCH_SIZE]
            video_paths = [manifest.at[i, "path"] for i in batch_idx]

            try:
                results = analyze_batch(video_paths)
                for idx, result in zip(batch_idx, results):
                    manifest.at[idx, "vlm_analysis"] = result
            except Exception as e:
                for idx in batch_idx:
                    manifest.at[idx, "vlm_analysis"] = f"ERROR: {e}"
                print(f"\nError on batch {[Path(p).name for p in video_paths]}: {e}")

            torch.cuda.empty_cache()
            processed += len(batch_idx)
            pbar.update(len(batch_idx))

            if processed % SAVE_EVERY < BATCH_SIZE:
                manifest.to_csv(MANIFEST_PATH, index=False)

    manifest.to_csv(MANIFEST_PATH, index=False)

# Final summary
total  = len(manifest)
done   = manifest["vlm_analysis"].notna().sum()
errors = manifest["vlm_analysis"].str.startswith("ERROR:", na=False).sum()
skips  = manifest["vlm_analysis"].str.startswith("SKIP:", na=False).sum()
print(f"\nFinal: {done}/{total}  |  {errors} errors  |  {skips} skips  |  {pass_num} retry pass(es).")
print(f"Results saved → {MANIFEST_PATH}")
manifest.head(3)
